# Movie Industry Analytics Lab NotebookThis notebook covers NumPy foundations, pandas loading, memory optimization, EDA, filtering, regex operations, and a full data quality audit for the integrated movie dataset.

In [ ]:
from pathlib import Pathimport sysimport pandas as pdPROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'docs' else Path.cwd()SRC_DIR = PROJECT_ROOT / 'src'if str(SRC_DIR) not in sys.path:    sys.path.append(str(SRC_DIR))from analytics.numpy_ops import run_numpy_foundationsfrom analytics.data_loader import (    load_available_mongo_collections,    load_csv_data,    optimize_dataframe_dtypes,    process_csv_in_chunks,    save_dataframe_csv,)from analytics.explorer import (    dataframe_info_text,    describe_dataframe,    extract_release_year,    inspect_structure,    save_distribution_charts,    value_counts_report,)from analytics.selector import (    filter_by_between,    filter_by_isin,    filter_quality_popularity,    filter_with_loc,    sample_with_iloc,    select_columns,)from analytics.regex_ops import (    count_crime_terms,    extract_numbers_from_titles,    extract_year_from_titles,    filter_titles_by_prefix,    most_common_genres,    short_overviews,    validate_ids,)from analytics.quality_report import run_full_quality_auditANALYTICS_DIR = PROJECT_ROOT / 'data' / 'processed' / 'analytics'CHARTS_DIR = ANALYTICS_DIR / 'charts'ANALYTICS_DIR.mkdir(parents=True, exist_ok=True)CHARTS_DIR.mkdir(parents=True, exist_ok=True)PROJECT_ROOT

## Part 1: NumPy Foundations

In [ ]:
numpy_result = run_numpy_foundations()print('Array metadata:')for name, meta in numpy_result['array_metadata'].items():    print(name, meta)print('\nVectorized metric sample:')print(numpy_result['vectorized_metrics']['weighted_score'])print('\nStatistics:', numpy_result['statistics'])

## Part 2: Load from MongoDB and Export CSV

In [ ]:
df_mongo = load_available_mongo_collections()print('Loaded shape:', df_mongo.shape)raw_csv_path = save_dataframe_csv(df_mongo, ANALYTICS_DIR / 'integrated_raw_export.csv')raw_csv_path

## Part 3: Chunked Processing and Memory Optimization

In [ ]:
df_csv = load_csv_data(raw_csv_path)rating_candidates = ['vote_average', 'rating_imdb', 'rating', 'score']language_candidates = ['original_language', 'language', 'metadata.language', 'source_collection']rating_col = next((c for c in rating_candidates if c in df_csv.columns), None)language_col = next((c for c in language_candidates if c in df_csv.columns), 'source_collection')if rating_col:    chunk_stats = process_csv_in_chunks(raw_csv_path, rating_col=rating_col, language_col=language_col, chunksize=1000)    print('Global mean:', chunk_stats['global_mean'])    print('Rating count:', chunk_stats['rating_count'])    pd.DataFrame(chunk_stats['per_language_mean'].items(), columns=['language', 'mean_rating']).head()else:    print('No rating column found for chunked global mean task.')optimized_df, mem_stats = optimize_dataframe_dtypes(df_csv)print(mem_stats)save_dataframe_csv(optimized_df, ANALYTICS_DIR / 'integrated_optimized.csv')

## Part 4: EDA and Distribution Charts

In [ ]:
release_date_col = next((c for c in ['release_date', 'metadata.release_date', 'date'] if c in optimized_df.columns), None)eda_df = extract_release_year(optimized_df, release_date_col) if release_date_col else optimized_df.copy()structure = inspect_structure(eda_df)print('Shape:', structure['shape'])print('Columns:', len(structure['columns']))info_text = dataframe_info_text(eda_df)(ANALYTICS_DIR / 'eda_info.txt').write_text(info_text, encoding='utf-8')desc = describe_dataframe(eda_df)desc['numeric'].to_csv(ANALYTICS_DIR / 'describe_numeric.csv')desc['categorical'].to_csv(ANALYTICS_DIR / 'describe_categorical.csv')cat_cols = [c for c in ['source_collection', 'original_language', 'language', 'status'] if c in eda_df.columns]value_reports = value_counts_report(eda_df, cat_cols, top_n=10)for col, vc in value_reports.items():    vc.rename_axis(col).reset_index(name='count').to_csv(ANALYTICS_DIR / f'value_counts_{col}.csv', index=False)chart_paths = save_distribution_charts(eda_df, CHARTS_DIR)chart_paths

## Part 5: loc, iloc, Boolean, isin, between

In [ ]:
selected = select_columns(eda_df, [c for c in ['title', 'name', rating_col, 'popularity', 'original_language', 'source_collection'] if c and c in eda_df.columns])iloc_sample = sample_with_iloc(eda_df, 0, 20, 2)if rating_col:    rating_series = pd.to_numeric(eda_df[rating_col], errors='coerce')    loc_result = filter_with_loc(eda_df, rating_series >= rating_series.median())else:    loc_result = eda_df.head(0)pop_col = next((c for c in ['popularity', 'vote_count', 'listeners'] if c in eda_df.columns), None)if rating_col and pop_col:    boolean_result = filter_quality_popularity(eda_df, rating_col, pop_col, min_rating=6.0, min_popularity=10.0)    between_result = filter_by_between(eda_df, rating_col, 4.0, 8.5)else:    boolean_result = eda_df.head(0)    between_result = eda_df.head(0)if language_col in eda_df.columns:    top_langs = eda_df[language_col].astype(str).value_counts().head(3).index.tolist()    isin_result = filter_by_isin(eda_df, language_col, top_langs)    isin_ex_result = filter_by_isin(eda_df, language_col, top_langs, exclude=True)else:    isin_result = eda_df.head(0)    isin_ex_result = eda_df.head(0)selected.head()

## Part 6: Regex Operations

In [ ]:
title_col = next((c for c in ['title', 'name', 'metadata.file_name'] if c in eda_df.columns), None)overview_col = next((c for c in ['overview', 'summary', 'description', 'content'] if c in eda_df.columns), None)id_col = next((c for c in ['id', 'movie_id', 'tmdb_id', 'imdb_id', 'metadata.id'] if c in eda_df.columns), None)regex_summary = pd.DataFrame(index=eda_df.index)if title_col:    title_series = eda_df[title_col].astype(str)    regex_summary['title_year'] = extract_year_from_titles(title_series)    regex_summary['title_numbers'] = extract_numbers_from_titles(title_series).astype(str)    regex_summary['prefix_the'] = title_series.index.isin(filter_titles_by_prefix(title_series, 'The').index)if overview_col:    overview_series = eda_df[overview_col].astype(str)    regex_summary['crime_term_count'] = count_crime_terms(overview_series)    regex_summary['short_overview'] = overview_series.index.isin(short_overviews(overview_series, max_words=8).index)genre_counts = most_common_genres(eda_df, top_n=10)id_validation = validate_ids(eda_df, id_col, source='tmdb') if id_col else pd.DataFrame()regex_summary.to_csv(ANALYTICS_DIR / 'regex_summary.csv', index=False)genre_counts.rename_axis('genre').reset_index(name='count').to_csv(ANALYTICS_DIR / 'regex_common_genres.csv', index=False)if not id_validation.empty:    id_validation.to_csv(ANALYTICS_DIR / 'regex_id_validation.csv', index=False)regex_summary.head()

## Part 7: Full Data Quality Report

In [ ]:
quality_output = run_full_quality_audit(    eda_df,    output_dir=ANALYTICS_DIR / 'quality',    id_columns=[c for c in ['id', 'movie_id', 'tmdb_id', 'imdb_id', 'metadata.id'] if c in eda_df.columns],    title_column=title_col or 'title',)quality_output

## Part 8: Optional Full Pipeline ExecutionIf you want to run everything from the script-based pipeline entrypoint:python src/run_pipeline.py